# SupplySenseAI — Technical Walkthrough

This notebook documents the complete SupplySenseAI pipeline: data preprocessing, quantile demand forecasting, conformal prediction, stock disruption detection, risk classification, order recommendations, and the Gemini-powered AI Copilot.

This notebook is environment-agnostic and runs in any standard Jupyter or Python environment. The production version of this system is deployed as a live Streamlit application (`app.py`) using the same logic found in the `src/` package.

**Dataset:** UCI Online Retail II (2009-2011 UK e-commerce transactions)

**AI Provider:** Google Gemini 2.0 Flash

## Step 1: Install and Import Libraries

In [ ]:
# Install required libraries (uncomment if running in a fresh environment)
# %pip install lightgbm mapie ruptures plotly scikit-learn openpyxl pandas numpy scipy requests python-dotenv -q

import warnings
warnings.filterwarnings("ignore")

import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import lightgbm as lgb
import ruptures as rpt
import requests
from scipy.stats import norm
from sklearn.metrics import mean_absolute_error, mean_squared_error

import mapie
_MAPIE_MAJOR = int(mapie.__version__.split(".")[0])
if _MAPIE_MAJOR >= 1:
    from mapie.regression import SplitConformalRegressor as MapieBackend
    MAPIE_NEW_API = True
else:
    from mapie.regression import MapieRegressor as MapieBackend
    MAPIE_NEW_API = False

print("Libraries imported successfully.")
print(f"LightGBM version : {lgb.__version__}")
print(f"MAPIE version    : {mapie.__version__}")
print(f"NumPy version    : {np.__version__}")
print(f"Pandas version   : {pd.__version__}")


## Step 2: Load Dataset

Place the **Online Retail II** dataset (CSV format) in a `data/` folder before running this notebook. The dataset can be obtained from the UCI Machine Learning Repository.

Expected path: `data/online_retail_II.csv`

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
DATA_FILE = DATA_DIR / "online_retail_II.csv"

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_FILE}. "
        "Download the Online Retail II dataset from the UCI Machine Learning "
        "Repository and place it at this path before continuing."
    )

df_raw = pd.read_csv(DATA_FILE, dtype={"CustomerID": str}, encoding="latin1")
df_raw.rename(columns={
    "Invoice": "InvoiceNo",
    "Price": "UnitPrice",
    "Customer ID": "CustomerID",
}, inplace=True)

print(f"Raw dataset shape: {df_raw.shape}")
print(f"Date range: {df_raw['InvoiceDate'].min()} to {df_raw['InvoiceDate'].max()}")
print(f"Unique products (StockCode): {df_raw['StockCode'].nunique()}")
print(f"Unique customers: {df_raw['CustomerID'].nunique()}")
df_raw.head()


### Data Quality Overview

In [ ]:
print("DATA QUALITY REPORT")
print(f"Total rows          : {len(df_raw):,}")
print(f"Missing Description  : {df_raw['Description'].isna().sum():,}")
print(f"Missing CustomerID   : {df_raw['CustomerID'].isna().sum():,}")
print(f"Negative Quantities  : {(df_raw['Quantity'] < 0).sum():,} (returns or cancellations)")
print(f"Negative Prices      : {(df_raw['UnitPrice'] <= 0).sum():,}")

df_raw["Revenue"] = df_raw["Quantity"] * df_raw["UnitPrice"]
top_products = (
    df_raw[df_raw["Revenue"] > 0]
    .groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print("\nTop 10 products by revenue:")
print(top_products.to_string())


## Step 3: Data Preprocessing and Feature Engineering

This logic mirrors `src/preprocessing/feature_engineering.py`.

In [ ]:
df = df_raw.copy()

# Remove cancellations and invalid rows
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]
df.dropna(subset=["Description", "CustomerID"], inplace=True)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Date"] = pd.to_datetime(df["InvoiceDate"].dt.date)

print(f"After cleaning: {len(df):,} rows")

# Aggregate to daily demand per SKU
daily = (
    df.groupby(["StockCode", "Date"])
      .agg(daily_qty=("Quantity", "sum"))
      .reset_index()
)

top_skus = (
    daily.groupby("StockCode")["daily_qty"]
         .sum()
         .sort_values(ascending=False)
         .head(50)
         .index.tolist()
)
daily = daily[daily["StockCode"].isin(top_skus)].copy()

all_dates = pd.date_range(daily["Date"].min(), daily["Date"].max(), freq="D")
full_index = pd.MultiIndex.from_product([top_skus, all_dates], names=["StockCode", "Date"])
daily = (
    daily.set_index(["StockCode", "Date"])
         .reindex(full_index, fill_value=0)
         .reset_index()
)

print(f"Daily demand matrix: {len(daily):,} rows ({len(top_skus)} SKUs x {len(all_dates)} days)")
daily.head()


In [ ]:
def engineer_features(df):
    df = df.sort_values(["StockCode", "Date"]).reset_index(drop=True)

    df["day_of_week"]    = df["Date"].dt.dayofweek
    df["day_of_month"]   = df["Date"].dt.day
    df["week_of_year"]   = df["Date"].dt.isocalendar().week.astype(int)
    df["month"]          = df["Date"].dt.month
    df["quarter"]        = df["Date"].dt.quarter
    df["is_weekend"]     = (df["day_of_week"] >= 5).astype(int)
    df["is_month_end"]   = df["Date"].dt.is_month_end.astype(int)
    df["is_month_start"] = df["Date"].dt.is_month_start.astype(int)

    grp = df.groupby("StockCode")["daily_qty"]
    for lag in [1, 7, 14, 28]:
        df[f"lag_{lag}"] = grp.shift(lag)

    for window in [7, 14, 28]:
        shifted = grp.shift(1)
        df[f"roll_mean_{window}"] = shifted.transform(lambda x: x.rolling(window, min_periods=1).mean())
        df[f"roll_std_{window}"]  = shifted.transform(lambda x: x.rolling(window, min_periods=1).std().fillna(0))
        df[f"roll_max_{window}"]  = shifted.transform(lambda x: x.rolling(window, min_periods=1).max())

    df["sku_code"] = df["StockCode"].astype("category").cat.codes
    df.dropna(subset=["lag_1", "lag_7", "lag_14", "lag_28"], inplace=True)
    return df

daily_feat = engineer_features(daily)

FEATURE_COLS = [
    "sku_code", "day_of_week", "day_of_month", "week_of_year",
    "month", "quarter", "is_weekend", "is_month_end", "is_month_start",
    "lag_1", "lag_7", "lag_14", "lag_28",
    "roll_mean_7", "roll_std_7", "roll_max_7",
    "roll_mean_14", "roll_std_14",
    "roll_mean_28", "roll_std_28", "roll_max_28",
]
TARGET_COL = "daily_qty"

print(f"Feature engineering complete: {daily_feat.shape}")
print(f"Features: {len(FEATURE_COLS)}")
print(f"Date range: {daily_feat[\'Date\'].min().date()} to {daily_feat[\'Date\'].max().date()}")


In [ ]:
dates_sorted = sorted(daily_feat["Date"].unique())
n = len(dates_sorted)

train_end = dates_sorted[int(n * 0.70)]
calib_end = dates_sorted[int(n * 0.85)]

train_df = daily_feat[daily_feat["Date"] <= train_end]
calib_df = daily_feat[(daily_feat["Date"] > train_end) & (daily_feat["Date"] <= calib_end)]
test_df  = daily_feat[daily_feat["Date"] > calib_end]

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
X_calib, y_calib = calib_df[FEATURE_COLS], calib_df[TARGET_COL]
X_test,  y_test  = test_df[FEATURE_COLS],  test_df[TARGET_COL]

print("Data split complete:")
print(f"  Train       : {len(X_train):>7,} rows ({train_df['Date'].min().date()} to {train_end.date()})")
print(f"  Calibration : {len(X_calib):>7,} rows ({calib_df['Date'].min().date()} to {calib_end.date()})")
print(f"  Test        : {len(X_test):>7,} rows ({test_df['Date'].min().date()} to {test_df['Date'].max().date()})")


## Step 4: Forecasting Engine — LightGBM Quantile Regression

Three models are trained: P10 (lower bound), P50 (median forecast), and P90 (upper bound). This mirrors `src/forecasting/lightgbm_model.py`.

In [ ]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

BASE_PARAMS = {
    "n_estimators": 800,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "n_jobs": -1,
    "verbosity": -1,
    "random_state": 42,
    "objective": "quantile",
}

lgbm_models = {}
for quantile, label in zip([0.10, 0.50, 0.90], ["p10", "p50", "p90"]):
    print(f"Training {label.upper()} model (alpha={quantile})...")
    params = {**BASE_PARAMS, "alpha": quantile}
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_calib, y_calib)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=30, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )
    lgbm_models[label] = model
    with open(MODEL_DIR / f"lgbm_{label}.pkl", "wb") as f:
        pickle.dump(model, f)
    print(f"  {label.upper()} trained and saved — best iteration: {model.best_iteration_}")

print("\nAll three quantile models trained and saved to models/.")


In [ ]:
def predict_quantiles(models, X):
    preds = {label: models[label].predict(X).clip(min=0) for label in ["p10", "p50", "p90"]}
    df = pd.DataFrame(preds, index=X.index)
    df["p10"] = df[["p10", "p50"]].min(axis=1)
    df["p90"] = df[["p50", "p90"]].max(axis=1)
    return df

q_preds_test = predict_quantiles(lgbm_models, X_test)
q_preds_test["y_true"]    = y_test.values
q_preds_test["Date"]      = test_df["Date"].values
q_preds_test["StockCode"] = test_df["StockCode"].values

cov_80 = ((q_preds_test["y_true"] >= q_preds_test["p10"]) &
          (q_preds_test["y_true"] <= q_preds_test["p90"])).mean()
mae  = mean_absolute_error(q_preds_test["y_true"], q_preds_test["p50"])
rmse = np.sqrt(mean_squared_error(q_preds_test["y_true"], q_preds_test["p50"]))

print("Quantile Regression Results:")
print(f"  MAE  (P50)       : {mae:.2f}")
print(f"  RMSE (P50)       : {rmse:.2f}")
print(f"  Coverage P10-P90 : {cov_80:.1%}  (target approximately 80%)")
q_preds_test.head()


## Step 5: Conformal Prediction (MAPIE)

Conformal prediction provides a mathematically guaranteed coverage level, independent of the underlying data distribution. This mirrors `src/uncertainty/conformal_prediction.py`.

In [ ]:
ALPHA = 0.05  # target 95% coverage
confidence = 1.0 - ALPHA

print("Fitting conformal predictor on the calibration set...")

if MAPIE_NEW_API:
    mapie_model = MapieBackend(
        estimator=lgbm_models["p50"],
        confidence_level=confidence,
        conformity_score="absolute",
        prefit=True,
    )
    mapie_model.conformalize(X_calib, y_calib)
    y_pred_mapie, intervals = mapie_model.predict_interval(X_test)
    conf_lower = np.clip(intervals[:, 0, 0], 0, None)
    conf_upper = intervals[:, 1, 0]
else:
    mapie_model = MapieBackend(
        estimator=lgbm_models["p50"],
        method="base",
        cv="prefit",
        random_state=42,
    )
    mapie_model.fit(X_calib, y_calib)
    y_pred_mapie, y_pis = mapie_model.predict(X_test, alpha=ALPHA)
    conf_lower = np.clip(y_pis[:, 0, 0], 0, None)
    conf_upper = y_pis[:, 1, 0]

print("Conformal calibration complete.")

q_preds_test["conf_lower"] = conf_lower
q_preds_test["conf_upper"] = conf_upper

in_interval   = ((q_preds_test["y_true"] >= q_preds_test["conf_lower"]) &
                  (q_preds_test["y_true"] <= q_preds_test["conf_upper"]))
empirical_cov = in_interval.mean()
calib_error   = abs(empirical_cov - confidence)
passes_gate   = calib_error <= 0.03

with open(MODEL_DIR / "mapie_model.pkl", "wb") as f:
    pickle.dump({"mapie": mapie_model, "alpha": ALPHA, "new_api": MAPIE_NEW_API}, f)

print("\nConformal Coverage Report:")
print(f"  Declared coverage  : {confidence:.0%}")
print(f"  Empirical coverage : {empirical_cov:.2%}")
print(f"  Calibration error  : {calib_error:.4f}")
print(f"  Passes +/-3% gate  : {\'PASS\' if passes_gate else \'FAIL\'}")


## Step 6: Stock Disruption Detection

A three-layer detection system identifies demand anomalies that may indicate supply chain disruptions: rolling Z-score anomalies, volatility regime shifts, and PELT change-point detection. This mirrors `src/risk_engine/stock_detector.py`.

In [ ]:
def detect_stock_disruption(series: pd.Series, z_threshold=3.0, vol_threshold=2.0):
    """Run full stock disruption detection on a single SKU demand series."""
    # Layer 1: Rolling Z-Score
    roll_mean = series.rolling(28, min_periods=7).mean()
    roll_std  = series.rolling(28, min_periods=7).std().replace(0, 1e-6)
    z_score   = (series - roll_mean) / roll_std
    anomalies = (z_score.abs() > z_threshold).astype(int)

    # Layer 2: Volatility Ratio
    std_short   = series.rolling(7,  min_periods=3).std().replace(0, 1e-6)
    std_long    = series.rolling(28, min_periods=7).std().replace(0, 1e-6)
    vol_ratio   = std_short / std_long
    is_volatile = (vol_ratio > vol_threshold).any()

    # Layer 3: PELT Change-Point Detection
    signal = series.values.astype(float)
    breakpoints = []
    if len(signal) >= 20:
        try:
            algo = rpt.Pelt(model="rbf", min_size=5, jump=1)
            algo.fit(signal)
            bps = algo.predict(pen=10.0)
            breakpoints = [bp for bp in bps if bp < len(signal)]
        except Exception:
            pass

    recent_anomalies     = int(anomalies.tail(14).sum())
    disruption_detected  = (recent_anomalies > 2) or is_volatile or (len(breakpoints) > 0)

    return {
        "disruption_detected": disruption_detected,
        "recent_anomalies":    recent_anomalies,
        "is_volatile":         bool(is_volatile),
        "changepoints":        breakpoints,
        "latest_zscore":       float(z_score.iloc[-1]) if len(z_score) > 0 else 0.0,
        "volatility_ratio":    float(vol_ratio.iloc[-1]) if len(vol_ratio) > 0 else 0.0,
    }

print("Running stock disruption detection across all SKUs...")
disruption_results = {}
for sku in top_skus:
    sku_series = daily_feat[daily_feat["StockCode"] == sku]["daily_qty"].reset_index(drop=True)
    disruption_results[sku] = detect_stock_disruption(sku_series)

n_disrupted = sum(1 for r in disruption_results.values() if r["disruption_detected"])
print(f"\nDetection complete:")
print(f"  Total SKUs analyzed     : {len(top_skus)}")
print(f"  SKUs with disruptions   : {n_disrupted} ({n_disrupted/len(top_skus):.0%})")
print(f"  Stable SKUs             : {len(top_skus) - n_disrupted}")

print("\nTop disrupted SKUs:")
for sku, res in disruption_results.items():
    if res["disruption_detected"]:
        print(f"  {str(sku)[:15]:<15} | Z-score: {res[\'latest_zscore\']:+.2f} | "
              f"Vol ratio: {res[\'volatility_ratio\']:.2f} | "
              f"Breakpoints: {len(res[\'changepoints\'])}")


## Step 7: Risk Classification Engine

Assigns LOW / MEDIUM / HIGH risk tiers based on forecast interval width, recent volatility, and stock disruption status. This mirrors `src/risk_engine/risk_classifier.py`.

In [ ]:
LOW_MEDIUM_THRESH  = 0.20
MEDIUM_HIGH_THRESH = 0.40

def compute_risk_score(p10, p50, p90, roll_std_7, roll_mean_28, w_interval=0.6, w_volatility=0.4):
    """
    Risk Score = w_interval * interval_width_ratio + w_volatility * volatility_score
    interval_width_ratio = (p90 - p10) / max(p50, 1)
    volatility_score      = roll_std_7 / max(roll_mean_28, 1)
    Normalized to [0, 1].
    """
    interval_cv  = (p90 - p10) / max(float(p50), 1.0)
    volatility_s = float(roll_std_7) / max(float(roll_mean_28), 1.0)
    raw = w_interval * interval_cv + w_volatility * volatility_s
    return min(raw / 2.0, 1.0)

def classify_risk(score, force_high=False):
    if force_high:
        return "HIGH"
    if score < LOW_MEDIUM_THRESH:
        return "LOW"
    if score < MEDIUM_HIGH_THRESH:
        return "MEDIUM"
    return "HIGH"

q_preds_test = q_preds_test.merge(
    test_df[["Date", "StockCode", "roll_std_7", "roll_mean_28"]].reset_index(drop=True),
    on=["Date", "StockCode"], how="left",
)

q_preds_test["force_high"] = q_preds_test["StockCode"].map(
    {s: r["disruption_detected"] and r["recent_anomalies"] > 3
     for s, r in disruption_results.items()}
).fillna(False)

q_preds_test["risk_score"] = q_preds_test.apply(
    lambda r: compute_risk_score(r["p10"], r["p50"], r["p90"],
                                  r.get("roll_std_7", 0), r.get("roll_mean_28", 1)),
    axis=1,
)
q_preds_test["risk_tier"] = q_preds_test.apply(
    lambda r: classify_risk(r["risk_score"], r["force_high"]), axis=1
)

counts = q_preds_test["risk_tier"].value_counts()
total  = len(q_preds_test)
print("Risk Classification Summary:")
for tier in ["LOW", "MEDIUM", "HIGH"]:
    c = counts.get(tier, 0)
    print(f"  {tier:<8} : {c:>6,} ({c/total:>5.1%})")

print(f"\nMean risk score: {q_preds_test[\'risk_score\'].mean():.3f}")
q_preds_test[["StockCode", "Date", "p10", "p50", "p90", "risk_score", "risk_tier"]].head(10)


## Step 8: Order Recommendation Engine

Calculates uncertainty-aware safety stock, reorder point, and stockout probability for each SKU. This mirrors `src/recommendation_engine/order_recommender.py`.

In [ ]:
LEAD_TIME      = 3  # days
SERVICE_LEVELS = {"LOW": 0.90, "MEDIUM": 0.95, "HIGH": 0.99}
Z_SCORES       = {tier: norm.ppf(sl) for tier, sl in SERVICE_LEVELS.items()}

def compute_demand_sigma(p10, p90):
    """Estimate demand standard deviation from the quantile spread."""
    return max((p90 - p10) / 2.5631, 0.01)

def compute_safety_stock(risk_tier, p10, p90, lead_time=LEAD_TIME):
    sigma = compute_demand_sigma(p10, p90)
    return max(Z_SCORES[risk_tier] * sigma * np.sqrt(lead_time), 0.0)

def compute_reorder_point(p50, safety_stock, lead_time=LEAD_TIME):
    return p50 * lead_time + safety_stock

def compute_stockout_prob(current_stock, rop, p10, p90, lead_time=LEAD_TIME):
    sigma_lt = compute_demand_sigma(p10, p90) * np.sqrt(lead_time)
    if sigma_lt < 1e-6:
        return 0.0 if current_stock >= rop else 1.0
    z = (current_stock - rop) / sigma_lt
    return max(0.0, min(1.0, float(1.0 - norm.cdf(z))))

latest = q_preds_test.sort_values("Date").groupby("StockCode").last().reset_index()
latest["current_stock"] = latest["p50"] * 2.5  # illustrative starting stock level

recs = []
for _, row in latest.iterrows():
    ss    = compute_safety_stock(row["risk_tier"], row["p10"], row["p90"])
    rop   = compute_reorder_point(row["p50"], ss)
    stock = row["current_stock"]
    sp    = compute_stockout_prob(stock, rop, row["p10"], row["p90"])
    needs_reorder = stock < rop

    recs.append({
        "StockCode":      row["StockCode"],
        "risk_tier":      row["risk_tier"],
        "risk_score":     round(row["risk_score"], 3),
        "p50_forecast":   round(row["p50"], 1),
        "safety_stock":   round(ss, 1),
        "reorder_point":  round(rop, 1),
        "current_stock":  round(stock, 1),
        "stockout_prob":  round(sp, 3),
        "needs_reorder":  needs_reorder,
        "action": (f"Order {max(rop - stock + ss, 0):.0f} units now"
                   if needs_reorder else "Monitor — no order needed"),
    })

rec_df = pd.DataFrame(recs).sort_values(
    ["risk_tier", "stockout_prob"],
    key=lambda x: x.map({"HIGH": 0, "MEDIUM": 1, "LOW": 2}) if x.name == "risk_tier" else x,
    ascending=[True, False],
)

print("Order Recommendations (Top 15):")
print(rec_df[["StockCode", "risk_tier", "p50_forecast", "safety_stock",
              "reorder_point", "stockout_prob", "action"]].head(15).to_string(index=False))


## Step 9: Evaluation and Calibration Metrics

This mirrors `src/evaluation/metrics.py`.

In [ ]:
y_true  = q_preds_test["y_true"].values
y_p10   = q_preds_test["p10"].values
y_p50   = q_preds_test["p50"].values
y_p90   = q_preds_test["p90"].values
c_lower = q_preds_test["conf_lower"].values
c_upper = q_preds_test["conf_upper"].values

def pinball(y, yhat, q):
    e = y - yhat
    return np.where(e >= 0, q * e, (q - 1) * e).mean()

def winkler_score(y, lo, hi, alpha=0.05):
    w = hi - lo
    pen_lo = (2 / alpha) * np.maximum(lo - y, 0)
    pen_hi = (2 / alpha) * np.maximum(y - hi, 0)
    return (w + pen_lo + pen_hi).mean()

mae_val   = mean_absolute_error(y_true, y_p50)
rmse_val  = np.sqrt(mean_squared_error(y_true, y_p50))
mape_val  = np.mean(np.abs((y_true[y_true > 0] - y_p50[y_true > 0]) / y_true[y_true > 0])) * 100
cov_80    = ((y_true >= y_p10) & (y_true <= y_p90)).mean()
cov_95    = ((y_true >= c_lower) & (y_true <= c_upper)).mean()
calib_err = abs(cov_95 - 0.95)
pb_p10    = pinball(y_true, y_p10, 0.10)
pb_p50    = pinball(y_true, y_p50, 0.50)
pb_p90    = pinball(y_true, y_p90, 0.90)
winkler   = winkler_score(y_true, c_lower, c_upper)
sharpness = (c_upper - c_lower).mean()

print("=" * 55)
print("       SUPPLYSENSEAI - EVALUATION REPORT")
print("=" * 55)
print("Point Forecast (P50)")
print(f"  MAE                    : {mae_val:>10.2f}")
print(f"  RMSE                   : {rmse_val:>10.2f}")
print(f"  MAPE                   : {mape_val:>10.1f}%")
print()
print("Quantile Calibration")
print(f"  Pinball Loss (P10)     : {pb_p10:>10.4f}")
print(f"  Pinball Loss (P50)     : {pb_p50:>10.4f}")
print(f"  Pinball Loss (P90)     : {pb_p90:>10.4f}")
print(f"  Coverage P10-P90       : {cov_80:>10.1%}  (target approximately 80%)")
print()
print("Conformal Prediction (95%)")
print(f"  Declared Coverage      :     95.00%")
print(f"  Empirical Coverage     : {cov_95:>10.2%}")
print(f"  Calibration Error      : {calib_err:>10.4f}")
print(f"  Passes +/-3% Gate      : {\'YES\' if calib_err <= 0.03 else \'NO\':>10}")
print(f"  Interval Sharpness     : {sharpness:>10.2f}  (mean width)")
print(f"  Winkler Score          : {winkler:>10.2f}  (lower is better)")
print("=" * 55)


### Predicted vs. Actual Values

In [ ]:
fig = px.scatter(
    q_preds_test, x="y_true", y="p50",
    title="Predicted (P50) vs. Actual Demand — Test Set",
    labels={"y_true": "Actual Demand", "p50": "Predicted Demand (P50)"},
    opacity=0.5, hover_data=["StockCode", "Date"],
)
max_val = max(q_preds_test["y_true"].max(), q_preds_test["p50"].max())
fig.add_shape(type="line", x0=0, y0=0, x1=max_val, y1=max_val,
              line=dict(color="red", dash="dash"))
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white", width=800, height=500)
fig.show()


## Step 10: Visualization Dashboard

Interactive Plotly charts equivalent to the live Streamlit dashboard.

In [ ]:
high_risk_skus = rec_df[rec_df["risk_tier"] == "HIGH"]["StockCode"].tolist()
demo_sku = high_risk_skus[0] if high_risk_skus else top_skus[0]
print(f"Showing forecast dashboard for SKU: {demo_sku}")

sku_df = q_preds_test[q_preds_test["StockCode"] == demo_sku].sort_values("Date")

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=sku_df["Date"], y=sku_df["y_true"],
                           name="Actual Demand", line=dict(color="#1f77b4", width=2)))
fig1.add_trace(go.Scatter(x=sku_df["Date"], y=sku_df["p50"],
                           name="P50 Forecast", line=dict(color="#ff7f0e", width=2, dash="dash")))
fig1.add_trace(go.Scatter(
    x=pd.concat([sku_df["Date"], sku_df["Date"][::-1]]),
    y=pd.concat([sku_df["p90"], sku_df["p10"][::-1]]),
    fill="toself", fillcolor="rgba(255,127,14,0.15)",
    line=dict(color="rgba(0,0,0,0)"), name="P10-P90 Quantile Band",
))
fig1.add_trace(go.Scatter(
    x=pd.concat([sku_df["Date"], sku_df["Date"][::-1]]),
    y=pd.concat([sku_df["conf_upper"], sku_df["conf_lower"][::-1]]),
    fill="toself", fillcolor="rgba(44,160,44,0.10)",
    line=dict(color="rgba(0,0,0,0)"), name="Conformal 95% Interval",
))
fig1.update_layout(
    title=f"Demand Forecast with Uncertainty Bands — SKU: {demo_sku}",
    xaxis_title="Date", yaxis_title="Daily Quantity",
    hovermode="x unified", height=420,
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(orientation="h", y=-0.2),
)
fig1.show()


In [ ]:
RISK_COLORS = {"LOW": "#28a745", "MEDIUM": "#ffc107", "HIGH": "#dc3545"}

fig2 = go.Figure()
for tier, color in RISK_COLORS.items():
    mask = sku_df["risk_tier"] == tier
    fig2.add_trace(go.Scatter(
        x=sku_df[mask]["Date"], y=sku_df[mask]["risk_score"],
        mode="markers+lines", name=tier,
        marker=dict(color=color, size=5), line=dict(color=color, width=1),
    ))

fig2.add_hline(y=LOW_MEDIUM_THRESH, line_dash="dot", line_color="orange",
               annotation_text="Medium threshold (0.20)")
fig2.add_hline(y=MEDIUM_HIGH_THRESH, line_dash="dot", line_color="red",
               annotation_text="High threshold (0.40)")
fig2.update_layout(
    title=f"Risk Score Timeline — SKU: {demo_sku}",
    xaxis_title="Date", yaxis_title="Risk Score",
    height=350, plot_bgcolor="white", paper_bgcolor="white",
)
fig2.show()


In [ ]:
risk_counts = q_preds_test["risk_tier"].value_counts()
fig3 = go.Figure(go.Pie(
    labels=risk_counts.index, values=risk_counts.values, hole=0.55,
    marker=dict(colors=["#28a745", "#ffc107", "#dc3545"]),
))
fig3.update_layout(
    title="Risk Tier Distribution — All SKUs (Test Set)", height=380,
    annotations=[dict(text="Risk<br>Tiers", x=0.5, y=0.5, font_size=16, showarrow=False)],
)
fig3.show()

q_preds_test["interval_width"] = q_preds_test["p90"] - q_preds_test["p10"]
sample = q_preds_test.sample(min(2000, len(q_preds_test)), random_state=42)

fig4 = px.scatter(
    sample, x="interval_width", y="risk_score", color="risk_tier", opacity=0.5,
    color_discrete_map=RISK_COLORS, title="Interval Width vs Risk Score",
    labels={"interval_width": "P10-P90 Interval Width", "risk_score": "Risk Score"},
    height=380,
)
fig4.update_layout(plot_bgcolor="white", paper_bgcolor="white")
fig4.show()


In [ ]:
top_risk = rec_df.head(15).copy()
top_risk["color"] = top_risk["risk_tier"].map(RISK_COLORS)

fig5 = go.Figure(go.Bar(
    x=top_risk["StockCode"], y=top_risk["stockout_prob"] * 100,
    marker_color=top_risk["color"],
    text=[f"{v:.1f}%" for v in top_risk["stockout_prob"] * 100],
    textposition="outside",
))
fig5.update_layout(
    title="Stockout Probability by SKU (Top 15 at risk)",
    xaxis_title="SKU", yaxis_title="Stockout Probability (%)",
    height=420, plot_bgcolor="white", paper_bgcolor="white", xaxis_tickangle=-40,
)
fig5.add_hline(y=5, line_dash="dot", line_color="red", annotation_text="5% alert threshold")
fig5.show()


In [ ]:
alphas, coverages, ideal_covs = np.arange(0.05, 1.0, 0.05), [], []

for a in alphas:
    if MAPIE_NEW_API:
        mapie_model.confidence_level = 1 - a
        _, intervals = mapie_model.predict_interval(X_test)
        lo = np.clip(intervals[:, 0, 0], 0, None)
        hi = intervals[:, 1, 0]
    else:
        _, pis = mapie_model.predict(X_test, alpha=a)
        lo = np.clip(pis[:, 0, 0], 0, None)
        hi = pis[:, 1, 0]
    cov = ((y_test.values >= lo) & (y_test.values <= hi)).mean()
    coverages.append(cov)
    ideal_covs.append(1 - a)

fig6 = go.Figure()
fig6.add_trace(go.Scatter(x=ideal_covs, y=coverages, mode="lines+markers",
                           name="SupplySenseAI", line=dict(color="#1f77b4", width=2)))
fig6.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                           name="Perfect Calibration", line=dict(color="red", dash="dash")))
fig6.update_layout(
    title="Calibration Curve — Conformal Prediction Coverage",
    xaxis_title="Declared Coverage Level", yaxis_title="Empirical Coverage",
    height=400, plot_bgcolor="white", paper_bgcolor="white",
)
fig6.show()
print("\nA well-calibrated model tracks the diagonal reference line closely.")


In [ ]:
sku_sim = q_preds_test[q_preds_test["StockCode"] == demo_sku].sort_values("Date").copy()

baseline_stock = sku_sim["p50"].mean() * 2.0  # flat safety buffer baseline
stockout_base  = (sku_sim["y_true"] > baseline_stock).mean()

sku_sim["dyn_ss"]    = sku_sim.apply(lambda r: compute_safety_stock(r["risk_tier"], r["p10"], r["p90"]), axis=1)
sku_sim["dyn_stock"] = sku_sim["p50"] * LEAD_TIME + sku_sim["dyn_ss"]
stockout_aware       = (sku_sim["y_true"] > sku_sim["dyn_stock"]).mean()

metrics       = ["Stockout Rate", "Avg Safety Stock", "Avg Stock Level"]
baseline_vals = [stockout_base * 100, baseline_stock * 0.3, baseline_stock]
aware_vals    = [stockout_aware * 100, sku_sim["dyn_ss"].mean(), sku_sim["dyn_stock"].mean()]

fig7 = go.Figure()
fig7.add_trace(go.Bar(name="Baseline (Point Forecast)", x=metrics, y=baseline_vals, marker_color="#adb5bd"))
fig7.add_trace(go.Bar(name="SupplySenseAI", x=metrics, y=aware_vals, marker_color="#1f77b4"))
fig7.update_layout(
    title="Simulation: Uncertainty-Aware vs Point Forecast Baseline",
    barmode="group", height=400, plot_bgcolor="white", paper_bgcolor="white", yaxis_title="Value",
)
fig7.show()

print(f"\nSimulation results for SKU {demo_sku}:")
print(f"  Stockout Rate -> Baseline: {stockout_base:.1%}  |  SupplySenseAI: {stockout_aware:.1%}")
print(f"  Improvement: {(stockout_base - stockout_aware) / max(stockout_base, 1e-6):.0%} reduction in stockouts")


## Step 11: AI Copilot — Google Gemini Integration

Ask supply chain questions in plain English. The copilot uses all forecast, risk, and recommendation data as context. This mirrors `src/copilot/ai_copilot.py` used by the live Streamlit application.

**Setup:** The API key is loaded from the `GEMINI_API_KEY` environment variable (via the `.env` file), with a hardcoded fallback for convenience. Get a free key at https://aistudio.google.com

In [ ]:
from dotenv import load_dotenv
load_dotenv()

GEMINI_API_KEY = os.environ.get(
    "GEMINI_API_KEY",
    "AQ.Ab8RN6JjAtMnTc4zwxyyZ4FJTTkh6hOqgOkQGAKl3eIozpj9Lg"
).strip()

GEMINI_MODEL    = "gemini-2.0-flash"
GEMINI_API_BASE = "https://generativelanguage.googleapis.com/v1beta/models"

SYSTEM_PROMPT = (
    "You are SupplySenseAI Copilot, an expert supply chain risk analyst. "
    "Help supply chain managers understand demand forecasts, uncertainty bands, "
    "and procurement recommendations in plain English. "
    "Be specific - reference exact numbers from the provided context. "
    "End every response with a clear, actionable recommendation. "
    "Keep answers concise: 4 to 6 sentences."
)

def build_sku_context(sku_code):
    """Build a context block for a given SKU from all computed results."""
    row         = latest[latest["StockCode"] == sku_code].iloc[0]
    rec         = rec_df[rec_df["StockCode"] == sku_code].iloc[0]
    disruption  = disruption_results.get(sku_code, {})
    return (
        f"SKU: {sku_code}\n"
        f"Daily Forecast: P10={row[\'p10\']:.1f}, P50={row[\'p50\']:.1f}, P90={row[\'p90\']:.1f}\n"
        f"Conformal 95% Interval: [{row[\'conf_lower\']:.1f}, {row[\'conf_upper\']:.1f}]\n"
        f"Risk Tier: {row[\'risk_tier\']}\n"
        f"Risk Score: {row[\'risk_score\']:.3f}\n"
        f"Safety Stock Recommendation: {rec[\'safety_stock\']:.0f} units\n"
        f"Reorder Point: {rec[\'reorder_point\']:.0f} units\n"
        f"Current Stock Level: {rec[\'current_stock\']:.0f} units\n"
        f"Stockout Probability: {rec[\'stockout_prob\']:.1%}\n"
        f"Stock Disruption Detected: {disruption.get(\'disruption_detected\', False)}\n"
        f"Recent Anomalies (14 days): {disruption.get(\'recent_anomalies\', 0)}\n"
        f"Latest Z-Score: {disruption.get(\'latest_zscore\', 0):.2f}\n"
        f"Volatility Ratio: {disruption.get(\'volatility_ratio\', 0):.2f}\n"
        f"Change Points Detected: {len(disruption.get(\'changepoints\', []))}"
    )

def ask_copilot(question, sku_code, history=None):
    """Send a question to Gemini with full SKU context and return (answer, updated_history)."""
    context = build_sku_context(sku_code)
    history = history or []

    contents = []
    for msg in history:
        gemini_role = "model" if msg["role"] == "assistant" else "user"
        contents.append({"role": gemini_role, "parts": [{"text": msg["content"]}]})

    user_text = f"SKU Context:\n{context}\n\nQuestion: {question}" if not history else question
    contents.append({"role": "user", "parts": [{"text": user_text}]})

    payload = {
        "system_instruction": {"parts": [{"text": f"{SYSTEM_PROMPT}\n\nSKU Context:\n{context}"}]},
        "contents": contents,
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.4},
    }

    url = f"{GEMINI_API_BASE}/{GEMINI_MODEL}:generateContent?key={GEMINI_API_KEY}"
    response = requests.post(url, headers={"Content-Type": "application/json"}, json=payload, timeout=30)

    if not response.ok:
        try:
            error_msg = response.json().get("error", {}).get("message", response.text)
        except Exception:
            error_msg = response.text
        raise RuntimeError(f"Gemini API error {response.status_code}: {error_msg}")

    data       = response.json()
    candidates = data.get("candidates", [])
    if not candidates:
        block_reason = data.get("promptFeedback", {}).get("blockReason", "unknown")
        raise RuntimeError(f"Gemini returned no candidates (blockReason: {block_reason})")

    answer = candidates[0]["content"]["parts"][0]["text"]

    updated_history = list(history)
    updated_history.append({"role": "user", "content": user_text})
    updated_history.append({"role": "assistant", "content": answer})
    return answer, updated_history

demo_questions = [
    "Why is this SKU flagged as high risk?",
    "What should I order next week and how much?",
    "Is the forecast reliable enough to trust?",
]

print(f"SupplySenseAI Copilot Demo — SKU: {demo_sku}")
print("=" * 60)

history = []
for q in demo_questions:
    print(f"\nYou: {q}")
    try:
        answer, history = ask_copilot(q, demo_sku, history)
    except Exception as e:
        answer = f"Copilot unavailable: {e}"
    print(f"AI: {answer}")
    print("-" * 60)


## Step 12: Project Summary and Key Results

In [ ]:
print("=" * 60)
print("     SUPPLYSENSEAI - FINAL PROJECT SUMMARY")
print("=" * 60)

print(f"\nDataset: UCI Online Retail II")
print(f"  SKUs analyzed         : {len(top_skus)}")
print(f"  Date range            : {daily_feat[\'Date\'].min().date()} to {daily_feat[\'Date\'].max().date()}")
print(f"  Daily demand records  : {len(daily_feat):,}")

print(f"\nModels Trained (LightGBM Quantile Regression)")
print(f"  P10 (lower bound)     : trained and saved")
print(f"  P50 (median)          : trained and saved")
print(f"  P90 (upper bound)     : trained and saved")

print(f"\nUncertainty Quantification")
print(f"  Conformal coverage    : {cov_95:.2%}  (target: 95%)")
print(f"  Calibration error     : {calib_err:.4f}")
print(f"  Gate (+/-3%) status   : {\'PASS\' if calib_err <= 0.03 else \'FAIL\'}")

print(f"\nRisk Classification")
risk_c = q_preds_test["risk_tier"].value_counts()
for t in ["LOW", "MEDIUM", "HIGH"]:
    print(f"  {t:<8} : {risk_c.get(t, 0):>6,} ({risk_c.get(t, 0)/len(q_preds_test):.1%})")

print(f"\nStock Disruption Detection")
print(f"  Disrupted SKUs        : {n_disrupted}/{len(top_skus)}")
print(f"  PELT algorithm        : Active")
print(f"  Z-score detection     : Active")
print(f"  Volatility monitor    : Active")

print(f"\nOrder Recommendations")
print(f"  SKUs needing reorder  : {rec_df[\'needs_reorder\'].sum()}/{len(rec_df)}")
print(f"  Avg stockout prob     : {rec_df[\'stockout_prob\'].mean():.1%}")

print(f"\nAI Copilot              : Active (Google Gemini 2.0 Flash)")
print("\n" + "=" * 60)
print("   SupplySenseAI — Pipeline run complete")
print("=" * 60)


## Next Step: Live Application

This notebook documents and validates the full pipeline. The same logic, packaged as reusable modules in `src/`, powers the live interactive application.

To launch the live Streamlit application:

```bash
streamlit run app.py
```

Or deploy directly to Streamlit Cloud by connecting this repository at [share.streamlit.io](https://share.streamlit.io) and adding `GEMINI_API_KEY` under Settings → Secrets.